# Stage C — tokenizer and CPU basal gate
Runs the five-tokenizer intrinsic study and tiny paper-MAC CPU comparison. Edit only the configuration cell.

In [ ]:
# USER CONFIGURATION
REPO_URL = 'https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF = '1cd65ffe7fee0d84da315594f36c58ac6a2333d5'
DRIVE_ROOT = '/content/drive/MyDrive/SeqTrainerStageC'
# Add the shared source-dataset folder to My Drive as a shortcut; do not copy its 15 Gbp contents.
SOURCE_DATASET_ROOT = '/content/drive/MyDrive/bacteria_titan_v1_ecoli_related_15gbp'
TRAIN_FASTA_DIR = f'{SOURCE_DATASET_ROOT}/shards/ecoli_related_gram_negative/train'
VALIDATION_FASTA_DIR = f'{SOURCE_DATASET_ROOT}/shards/ecoli_related_gram_negative/val'
RUN_NAME = 'c1_tokenizers_cpu'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
train_fastas = sorted(str(path) for path in Path(TRAIN_FASTA_DIR).glob('*.fa.gz'))
validation_fastas = sorted(str(path) for path in Path(VALIDATION_FASTA_DIR).glob('*.fa.gz'))
if not train_fastas or not validation_fastas:
    raise FileNotFoundError(
        'Stage C FASTA shards were not found. In Drive, add the shared '
        'bacteria_titan_v1_ecoli_related_15gbp folder as a shortcut to My Drive, '
        'then confirm SOURCE_DATASET_ROOT names that shortcut.'
    )
import subprocess, sys
repo = Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)], check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL], check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'], check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan,evo2-tokenizer]'], check=True)
print(subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip())

In [ ]:
from huggingface_hub import snapshot_download
dnabert = f'{DRIVE_ROOT}/tokenizers/dnabert2'
snapshot_download('zhihan1996/DNABERT-2-117M', local_dir=dnabert, allow_patterns=['tokenizer*','vocab*','special_tokens_map.json','tokenizer_config.json','config.json'])
output = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
def logged(label, command):
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',label,'--repo',str(repo),'--',*command], check=True)
def fasta_arguments(flag, paths):
    return [value for path in paths for value in (flag, path)]
train_arguments = fasta_arguments('--train-fasta', train_fastas)
validation_arguments = fasta_arguments('--validation-fasta', validation_fastas)
logged('tokenizer_study',['seqtrainer-titans-stage-c-tokenizers',*train_arguments,*validation_arguments,'--dnabert2-path',dnabert,'--output-dir',output])
logged('cpu_pilot',['seqtrainer-titans-stage-c-cpu-pilot',*train_arguments,*validation_arguments,'--dnabert2-path',dnabert,'--output-dir',output,'--steps','3'])

In [ ]:
import os
print('SHARE THIS DIRECTORY:', output)
print('\n'.join(sorted(os.listdir(output))))